# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mominullptr/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---

### Finding 1 — "Content items flagged by FlyRank for refresh show measurably higher subsequent engagement recovery than unflagged items"

**What the paper claims:** Pages that received a FlyRank refresh recommendation and were acted upon showed a directional lift in search visibility metrics (impressions and clicks) in the weeks following the editorial update, relative to a control group of comparable unflagged pages.

**My methodology question — where does the label come from?**
The evaluation label here is a *post-intervention* engagement recovery metric, derived from the same GSC impression and click data stream used to define the original decay signal. The key question I would ask is: *is the comparison group (unflagged pages) truly comparable on pre-treatment baseline characteristics?* If FlyRank's system preferentially flags the highest-volume, highest-rank pages for refresh, then any observed recovery could partly reflect **regression to the mean** — large pages that dipped temporarily recover naturally even without editorial action. The paper would be strengthened by a matched-cohort design that controls for baseline impression volume, ranking tier, and content age before attributing recovery to the refresh action. As currently disclosed, this finding is best read as a directional, observational association rather than a causal treatment effect.

**My methodology question — does the validation design carry the claim?**
The study's window design (observation → intervention → outcome) is coherent in direction, but the length of the post-intervention measurement window matters significantly. A 14-day outcome window may capture algorithmic re-indexing noise; a 60-day window is more stable but may confound with seasonal search demand shifts. I would ask: *was the outcome window pre-registered or chosen after observing the data?* Selecting the window that shows the best lift would inflate confidence. A constructive improvement would be reporting lift curves across multiple lag windows (14d, 30d, 60d) to show whether the recovery is durable or a short-term rebound artefact.

---

### Finding 2 — "The ML scoring model achieves substantially higher Precision@50 than the deterministic heuristic baseline across holdout clients"

**What the paper claims:** A trained ML model (described as Random Forest or similar ensemble) outperforms a rule-based baseline by a substantial margin on the Precision@K ranking metric evaluated on held-out client data, demonstrating that learned multi-signal scoring generalises beyond simple threshold rules.

**My methodology question — where does the label come from?**
The paper uses `is_declining_label` derived from the trailing 30-day vs prior 30-day impression comparison (`trend_direction == "down"`). This proxy label is computed from the **same 90-day window** used to extract features such as `impressions_90d`, `clicks_90d`, and `days_with_impressions`. The critical question is: *does the 90-day aggregate feature window temporally overlap the label's 30-day measurement window?* If `impressions_90d` includes the final month in which the decline was measured, then a portion of the feature directly encodes the label outcome — a form of window-overlap leakage. The paper should explicitly diagram the feature extraction window vs the label measurement window and confirm that `impressions_90d` is computed over a window that *ends before* the label's observation period, or that only `*_prev_30d`-style features are used when the label derives from the last 30-day delta.

**My methodology question — does the validation design carry the claim?**
The grouped client-holdout split (holding out 20% of clients unseen during training) is the methodologically correct design for this setting and I commend its use. The residual question is: *how many distinct holdout clients are in the test set, and how variable is Precision@50 across them?* If 6 clients are held out and the metric is averaged across them, a single high-volume client with a concentrated decay pattern could dominate the aggregate score. Reporting per-client Precision@50 distribution (median and interquartile range across holdout clients) would make the claim more robust and reproducible, and would help practitioners understand whether the model generalises uniformly or is driven by a few easy clients.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load data — same source used in Week 4 baseline and Week 5 model
raw_path = Path("data/raw/content_refresh_anonymized.csv")
if not raw_path.exists():
    raw_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(raw_path)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Demonstrate the window-overlap concern from Finding 2
# The label is derived from: trend_direction = "down" when
#   (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d < -0.20
# The feature impressions_90d covers the full 90-day window, which INCLUDES the last 30 days
# This is the temporal overlap we flag in Section 1.

label_col = "is_declining_label"
leakage_suspects = ["trend_direction", "trend_pct", "impressions_last_30d",
                    "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
                    "sessions_last_30d", "sessions_prev_30d"]

overlap_check = {col: (col in df.columns) for col in leakage_suspects}

print("=" * 75)
print("SECTION 1 — WINDOW OVERLAP VERIFICATION: LABEL-SOURCE COLUMNS")
print("=" * 75)
print("Label definition: is_declining_label = 1 when trend_direction == 'down'")
print("  i.e. (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d < -0.20")
print()
print("Label-source columns present in dataset:")
for col, present in overlap_check.items():
    status = "PRESENT (EXCLUDED from features)" if present else "not found"
    print(f"  {col:<32}: {status}")

print()
print("Correlation of impressions_90d with is_declining_label (full 90d window):")
corr_90d = df["impressions_90d"].corr(df["is_declining_label"])
print(f"  Pearson r = {corr_90d:.4f} (low -> no simple window-leakage via this channel)")

print()
print("Correlation of impressions_last_30d with is_declining_label (leaky window):")
corr_leaky = df["impressions_last_30d"].corr(df["is_declining_label"])
print(f"  Pearson r = {corr_leaky:.4f} (higher abs value confirms label-source leakage risk)")
print("=" * 75)


SECTION 1 — WINDOW OVERLAP VERIFICATION: LABEL-SOURCE COLUMNS
Label definition: is_declining_label = 1 when trend_direction == 'down'
  i.e. (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d < -0.20

Label-source columns present in dataset:
  trend_direction                 : PRESENT (EXCLUDED from features)
  trend_pct                       : PRESENT (EXCLUDED from features)
  impressions_last_30d            : PRESENT (EXCLUDED from features)
  impressions_prev_30d            : PRESENT (EXCLUDED from features)
  clicks_last_30d                 : PRESENT (EXCLUDED from features)
  clicks_prev_30d                 : PRESENT (EXCLUDED from features)
  sessions_last_30d               : PRESENT (EXCLUDED from features)
  sessions_prev_30d               : PRESENT (EXCLUDED from features)

Correlation of impressions_90d with is_declining_label (full 90d window):
  Pearson r = -0.0182 (low -> no simple window-leakage via this channel)

Correlation of impressions_last_30d wit

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before/After Split Design Comparison

**The Concern Being Tested:**
In Week 5 we already used a grouped client-holdout split. The question this section answers is: *how much does the metric shift when we move from a naive random row split (the tempting default) to the grouped client split (the honest design)?* The gap between the two numbers measures how much domain memorisation was occurring.

**Split Designs:**
1. **Naive Random Row Split (80/20, stratified)** — allows pages from the same client to appear in both train and test. This is the default most beginners use and the split most likely to inflate metrics.
2. **Grouped Client-Holdout Split (seed=42, 6 holdout clients)** — the honest split used in Week 5. Pages from any given client appear in exactly one partition.

**Expected Finding:** The grouped split produces lower Precision@K than the random split. The gap quantifies how much skill was "borrowed" from client-level memorisation under the naive split.

**Interpretation of the Gap:**
- A large gap (>15 pp on P@50) would indicate that the model was significantly relying on client identity patterns rather than genuine signal generalisation.
- A modest gap (<10 pp) suggests the learned signals (search visibility, CTR, content age) transfer reasonably across unseen clients, which is the directional finding we observe here.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# ── Feature matrix (identical to Week-5 model, pre-decision signals only) ──
numeric_features = [
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "content_age_days",
    "days_with_impressions", "days_with_sessions",
    "sessions_90d", "engaged_sessions_90d", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "word_count", "char_count",
    "search_volume", "competition", "cpc"
]
categorical_features = ["content_type", "main_intent", "competition_level",
                        "position_tier", "freshness_tier"]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("missing"), drop_first=True, dtype=float)
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"].values

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

RF_PARAMS = dict(n_estimators=200, max_depth=10, min_samples_leaf=25,
                 class_weight="balanced_subsample", random_state=42, n_jobs=-1)

# ── DESIGN A: Naive random row split (dishonest default) ──────────────────
all_idx = np.arange(len(df))
train_idx_rnd, test_idx_rnd = train_test_split(
    all_idx, test_size=0.2, random_state=42, stratify=y)

rf_rnd = RandomForestClassifier(**RF_PARAMS)
rf_rnd.fit(X.iloc[train_idx_rnd], y[train_idx_rnd])
probs_rnd = rf_rnd.predict_proba(X.iloc[test_idx_rnd])[:, 1]

rnd_p20  = precision_at_k(y[test_idx_rnd], probs_rnd, 20)
rnd_p50  = precision_at_k(y[test_idx_rnd], probs_rnd, 50)
rnd_p100 = precision_at_k(y[test_idx_rnd], probs_rnd, 100)
rnd_auc  = roc_auc_score(y[test_idx_rnd], probs_rnd)
rnd_br   = y[test_idx_rnd].mean()

# ── DESIGN B: Grouped client-holdout split (honest split, Week-5 design) ──
clients = df["client_id"].unique()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])

test_mask  = df["client_id"].isin(test_clients).values
train_mask = ~test_mask

rf_grp = RandomForestClassifier(**RF_PARAMS)
rf_grp.fit(X[train_mask], y[train_mask])
probs_grp = rf_grp.predict_proba(X[test_mask])[:, 1]

grp_p20  = precision_at_k(y[test_mask], probs_grp, 20)
grp_p50  = precision_at_k(y[test_mask], probs_grp, 50)
grp_p100 = precision_at_k(y[test_mask], probs_grp, 100)
grp_auc  = roc_auc_score(y[test_mask], probs_grp)
grp_br   = y[test_mask].mean()

# ── Summary comparison table ───────────────────────────────────────────────
print("=" * 90)
print("BEFORE/AFTER SPLIT DESIGN COMPARISON (Random Forest, same hyperparameters)")
print("=" * 90)
rows = [
    {"Metric": "Test Base Decay Rate",       "Random Row Split (Before)": f"{rnd_br*100:.1f}%",  "Grouped Client-Holdout (After)": f"{grp_br*100:.1f}%",  "Gap": f"{(rnd_br-grp_br)*100:+.1f} pp"},
    {"Metric": "Test Set Size",              "Random Row Split (Before)": f"{len(test_idx_rnd):,} rows (6,000)", "Grouped Client-Holdout (After)": f"{test_mask.sum():,} rows (6 clients)", "Gap": "—"},
    {"Metric": "Precision@20",               "Random Row Split (Before)": f"{rnd_p20*100:.1f}%",  "Grouped Client-Holdout (After)": f"{grp_p20*100:.1f}%",  "Gap": f"{(grp_p20-rnd_p20)*100:+.1f} pp"},
    {"Metric": "Precision@50 (primary)",     "Random Row Split (Before)": f"{rnd_p50*100:.1f}%",  "Grouped Client-Holdout (After)": f"{grp_p50*100:.1f}%",  "Gap": f"{(grp_p50-rnd_p50)*100:+.1f} pp"},
    {"Metric": "Precision@100",              "Random Row Split (Before)": f"{rnd_p100*100:.1f}%", "Grouped Client-Holdout (After)": f"{grp_p100*100:.1f}%", "Gap": f"{(grp_p100-rnd_p100)*100:+.1f} pp"},
    {"Metric": "ROC-AUC",                    "Random Row Split (Before)": f"{rnd_auc:.3f}",        "Grouped Client-Holdout (After)": f"{grp_auc:.3f}",        "Gap": f"{grp_auc-rnd_auc:+.3f}"},
]
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Interpretation:")
print(f"  P@50 shift from random to grouped split: {(grp_p50-rnd_p50)*100:+.1f} pp")
print(f"  A negative gap would indicate random-split inflation from client memorisation.")
print(f"  Both splits use identical features, hyperparameters, and random seed.")
print("=" * 90)


BEFORE/AFTER SPLIT DESIGN COMPARISON (Random Forest, same hyperparameters)
                Metric Random Row Split (Before) Grouped Client-Holdout (After)      Gap
  Test Base Decay Rate                     54.2%                          39.1% +15.1 pp
         Test Set Size        6,000 rows (6,000)         2,325 rows (6 clients)        —
          Precision@20                     90.0%                          90.0%  +0.0 pp
Precision@50 (primary)                     86.0%                          84.0%  -2.0 pp
         Precision@100                     88.0%                          82.0%  -6.0 pp
               ROC-AUC                     0.757                          0.757   -0.000

Interpretation:
  P@50 shift from random to grouped split: -2.0 pp
  A negative gap would indicate random-split inflation from client memorisation.
  Both splits use identical features, hyperparameters, and random seed.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Attack Checklist

We audit the Week-5 feature matrix against all three leakage types from the `hunting-leakage-and-validating` skill:

**Type 1 — Label-derived features:**
The label `is_declining_label` is derived directly from `trend_direction`, which is itself computed from `trend_pct` using `impressions_last_30d` and `impressions_prev_30d`. None of these four columns appear in our feature matrix — confirmed below.

**Type 2 — Future/overlapping window features:**
Our features are all 90-day trailing aggregates measured prior to the label snapshot. The label's signal (`trend_direction`) compares the last 30 days to the prior 30 days within the same snapshot. Since we exclude `impressions_last_30d` and `impressions_prev_30d` entirely, and use only `impressions_90d` (the full-window aggregate), there is no direct numerical leakage. However, `impressions_90d` *does* partially overlap the label window — we disclose this as a limitation.

**Type 3 — Decision-derived product flags:**
FlyRank's own refresh flags, health scores, and CTR-fix flags are not present in the starter dataset (they are stripped before delivery). No product-flag leakage is possible.

**The Deliberate Injection Test (Proof of Test Harness):**
As required by the skill checklist, we deliberately inject a known-leaky feature (`trend_pct`, the exact label source) and confirm the score jumps toward 1.0. Then we remove it and keep the honest number.

In [3]:
# ── 1. Confirm forbidden columns are absent from feature set ────────────────
forbidden_cols = ["trend_direction", "trend_pct", "impressions_last_30d",
                  "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
                  "sessions_last_30d", "sessions_prev_30d", "is_declining_label"]

feature_col_names = list(X.columns)
leakage_in_features = [c for c in forbidden_cols if c in feature_col_names]

print("=" * 75)
print("LEAKAGE AUDIT — FEATURE MATRIX COLUMN INSPECTION")
print("=" * 75)
print(f"Total feature dimensions  : {len(feature_col_names)}")
print(f"Forbidden columns checked : {len(forbidden_cols)}")
print(f"Leakage columns detected  : {len(leakage_in_features)}")
if leakage_in_features:
    print(f"  FAIL: {leakage_in_features}")
else:
    print("  PASS: No forbidden columns found in feature matrix.")

# ── 2. Deliberate injection test: add trend_pct (the label formula numerator) ─
X_leaky = X.copy()
X_leaky["trend_pct_INJECTED"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)

rf_leaky = RandomForestClassifier(**RF_PARAMS)
rf_leaky.fit(X_leaky[train_mask], y[train_mask])
probs_leaky = rf_leaky.predict_proba(X_leaky[test_mask])[:, 1]
leaky_p50 = precision_at_k(y[test_mask], probs_leaky, 50)
leaky_auc = roc_auc_score(y[test_mask], probs_leaky)

print()
print("=" * 75)
print("DELIBERATE INJECTION TEST: trend_pct added as feature (label numerator)")
print("=" * 75)
print(f"  Honest model  P@50: {grp_p50*100:.1f}%  | ROC-AUC: {grp_auc:.3f}")
print(f"  Leaky  model  P@50: {leaky_p50*100:.1f}%  | ROC-AUC: {leaky_auc:.3f}")
print(f"  Score jump on P@50: {(leaky_p50 - grp_p50)*100:+.1f} pp")
print()
print("  Verdict: A large positive jump confirms the test harness works and")
print("  that trend_pct is a genuine label source — it must stay excluded.")
print("  The honest number (without injection) is the one we report.")

# ── 3. Top feature importance check — confirm no suspiciously perfect signal ─
honest_importances = pd.Series(
    rf_grp.feature_importances_, index=X.columns
).sort_values(ascending=False)

print()
print("=" * 75)
print("TOP 10 MDI FEATURE IMPORTANCES (HONEST MODEL)")
print("=" * 75)
for feat, imp in honest_importances.head(10).items():
    flag = " <- INVESTIGATE if > 0.40" if imp > 0.40 else ""
    print(f"  {feat:<30}: {imp:.4f}{flag}")
print()
top_imp = honest_importances.iloc[0]
print(f"  Max single-feature importance: {top_imp:.4f}")
print(f"  Verdict: {'SUSPICIOUS (>40%)' if top_imp > 0.40 else 'ACCEPTABLE — No single feature dominates; no label leakage signature.'}")
print("=" * 75)


LEAKAGE AUDIT — FEATURE MATRIX COLUMN INSPECTION
Total feature dimensions  : 34
Forbidden columns checked : 9
Leakage columns detected  : 0
  PASS: No forbidden columns found in feature matrix.



DELIBERATE INJECTION TEST: trend_pct added as feature (label numerator)
  Honest model  P@50: 84.0%  | ROC-AUC: 0.757
  Leaky  model  P@50: 100.0%  | ROC-AUC: 1.000
  Score jump on P@50: +16.0 pp

  Verdict: A large positive jump confirms the test harness works and
  that trend_pct is a genuine label source — it must stay excluded.
  The honest number (without injection) is the one we report.

TOP 10 MDI FEATURE IMPORTANCES (HONEST MODEL)
  days_with_impressions         : 0.1572
  impressions_90d               : 0.1376
  avg_position                  : 0.1249
  content_age_days              : 0.1240
  word_count                    : 0.0553
  char_count                    : 0.0526
  scroll_rate                   : 0.0372
  clicks_90d                    : 0.0359
  ctr                           : 0.0354
  days_since_last_update        : 0.0328

  Max single-feature importance: 0.1572
  Verdict: ACCEPTABLE — No single feature dominates; no label leakage signature.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Bold Claims from Week 5 (w05_model.ipynb)

The following sentences from the Week-5 notebook go further than the evidence strictly supports. Each is rewritten below using safe claim language.

---

**Original Claim A** (from Section 3 markdown):
> *"Random Forest achieving 84.0% Precision@50 (+60.0 pp lift)"*

**Problem:** The number is stated as a bare fact without noting (a) the specific holdout partition it was computed on, (b) the test base rate of 39.1% (vs the training base rate of 55.5%), or (c) that a 6-client holdout is a small sample across which performance may vary.

**Rewritten (safe) Claim A:**
> *"On the 6-client holdout partition used in this evaluation (2,325 URLs, 39.1% observed decay rate), the Random Forest model produced a measured Precision@50 of 84.0%, compared to 24.0% for the deterministic heuristic baseline on the same partition. This directional improvement ($+60$ pp) suggests that learned multi-signal scoring may generalise beyond simple threshold rules in this dataset context. Results are from a single grouped-client split and should be interpreted as indicative rather than definitively generalisable to all content portfolios."*

---

**Original Claim B** (from Section 4 markdown):
> *"The model achieves >85% precision on Page 1 and Page 2 visible pages (impressions >= 500)"*

**Problem:** This was asserted in the narrative without being directly backed by a computed number in the notebook itself — it was an interpolation from the full-test-set metrics, not a verified sub-group result.

**Rewritten (safe) Claim B:**
> *"On the holdout test partition, the model's top-ranked candidates were observationally concentrated among high-visibility pages (impressions >= 500). The subgroup precision for this slice is computed and reported in the code cell below. This number is directional evidence that the model prioritises high-traffic decay risk, but the small sample size within any single position tier means individual tier results carry wide uncertainty."*

---

**Original Claim C** (from Section 1 markdown):
> *"Gradient Boosting achieving 88.0% Precision@50 (+64.0 pp lift)"*

**Problem:** Gradient Boosting edged out Random Forest on this specific holdout partition, but the margin (4 pp) is within the noise of a 50-item precision estimate. Stating the higher number without noting the closeness implies a reliability the data does not support.

**Rewritten (safe) Claim C:**
> *"Gradient Boosting and Random Forest both measured Precision@50 in the 84–88% range on this holdout partition, representing a directional lift of roughly 60 pp over the heuristic baseline. The 4 pp difference between the two ensembles falls within the expected sampling variability of a 50-item precision estimate and should not be interpreted as a reliable performance ranking between these two models. Both are meaningfully stronger than the baseline for decision-support triage."*

In [4]:
# ── Verify Claim B: sub-group precision on high-visibility pages ─────────────
df_test_audit = df[test_mask].copy()
df_test_audit["rf_prob"] = probs_grp

# High-visibility slice: impressions_90d >= 500 AND avg_position > 0 (ranked pages)
high_vis_mask = (df_test_audit["impressions_90d"] >= 500) & (df_test_audit["avg_position"] > 0)
tail_mask     = (df_test_audit["impressions_90d"] < 100)

def subgroup_p_at_k(sub_df, k=50):
    sub = sub_df.sort_values("rf_prob", ascending=False).head(k)
    if len(sub) < k:
        return None, len(sub)
    return float(sub["is_declining_label"].mean()), k

hv_p50, hv_k   = subgroup_p_at_k(df_test_audit[high_vis_mask], k=50)
tail_p50, tail_k = subgroup_p_at_k(df_test_audit[tail_mask], k=50)
full_p50, _     = subgroup_p_at_k(df_test_audit, k=50)

print("=" * 80)
print("CLAIM B VERIFICATION: Subgroup Precision@50 by Traffic Tier")
print("=" * 80)
print(f"  Full holdout test (2,325 URLs)    : P@50 = {full_p50*100:.1f}%")
print(f"  High-visibility pages (Imp >= 500): P@50 = {hv_p50*100:.1f}%  (n={high_vis_mask.sum():,})")
print(f"  Tail pages (Imp < 100)            : P@50 = {tail_p50*100:.1f}%  (n={tail_mask.sum():,})" if tail_p50 is not None else f"  Tail pages (Imp < 100)            : fewer than 50 pages available (n={tail_mask.sum():,})")
print()
print("  Interpretation: High-visibility pages show directionally higher precision.")
print("  This supports the claim that the model concentrates decay risk among")
print("  ranked, high-traffic pages — but sample sizes per tier are modest.")

# ── Claim C: margin between RF and GBM ─────────────────────────────────────
from sklearn.ensemble import HistGradientBoostingClassifier
gbm = HistGradientBoostingClassifier(max_iter=100, max_depth=5, min_samples_leaf=25,
                                     class_weight="balanced", random_state=42)
gbm.fit(X[train_mask], y[train_mask])
probs_gbm = gbm.predict_proba(X[test_mask])[:, 1]
gbm_p50 = precision_at_k(y[test_mask], probs_gbm, 50)

margin = abs(gbm_p50 - grp_p50) * 100
p50_ci_half = 100 * np.sqrt(grp_p50 * (1 - grp_p50) / 50)  # approx binomial SE * 1.96
print()
print("=" * 80)
print("CLAIM C VERIFICATION: RF vs GBM Precision@50 margin")
print("=" * 80)
print(f"  Random Forest  P@50 : {grp_p50*100:.1f}%")
print(f"  Gradient Boost P@50 : {gbm_p50*100:.1f}%")
print(f"  Observed margin     : {margin:.1f} pp")
print(f"  Approx 95% CI half  : +/- {p50_ci_half:.1f} pp (binomial, n=50)")
print(f"  Verdict: Margin ({margin:.1f} pp) {'is WITHIN' if margin < p50_ci_half*2 else 'EXCEEDS'} "
      f"the 95% CI ({p50_ci_half*2:.1f} pp) — {'difference is noise.' if margin < p50_ci_half*2 else 'marginally significant.'}")
print("=" * 80)


CLAIM B VERIFICATION: Subgroup Precision@50 by Traffic Tier
  Full holdout test (2,325 URLs)    : P@50 = 84.0%
  High-visibility pages (Imp >= 500): P@50 = 86.0%  (n=618)
  Tail pages (Imp < 100)            : P@50 = 76.0%  (n=1,430)

  Interpretation: High-visibility pages show directionally higher precision.
  This supports the claim that the model concentrates decay risk among
  ranked, high-traffic pages — but sample sizes per tier are modest.



CLAIM C VERIFICATION: RF vs GBM Precision@50 margin
  Random Forest  P@50 : 84.0%
  Gradient Boost P@50 : 88.0%
  Observed margin     : 4.0 pp
  Approx 95% CI half  : +/- 5.2 pp (binomial, n=50)
  Verdict: Margin (4.0 pp) is WITHIN the 95% CI (10.4 pp) — difference is noise.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.